# T0.8c: wheelhouse-first environment (replaces T0.8b)

**What T0.8b taught us:** the competition wheelhouse does not contain every transitive dependency (for example `typing-inspection`, `attrs`), so a strict pip resolution fails. The real sandbox image supplies those from its base environment. This notebook therefore picks the highest compatible wheel for each requirement itself, installs them with `--no-deps` into a per-task folder that goes FIRST on `PYTHONPATH`, and lets Kaggle's own environment supply anything the wheelhouse lacks.

**Settings:** Accelerator None, Internet Off. **Add Input:** main competition + `darkedges-data`. **Run All.** Paste me the last cell's output (and the output of the diagnostics cell if it errors).

In [ ]:
import glob, json, os, re, shlex, shutil, subprocess, sys, time
from pathlib import Path
COMP = "/kaggle/input/competitions/gemma-4-developer-agent"
if not os.path.exists(f"{COMP}/tasks.jsonl"):
    hits = glob.glob("/kaggle/input/**/tasks.jsonl", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else None
assert COMP, "MAIN competition not attached: Add Input -> Competitions -> 'Gemma 4 Developer Agent' (not the paper track)."
hits = glob.glob("/kaggle/input/**/tracelens/tracer.py", recursive=True)
assert hits, "Attach your private dataset 'darkedges-data': Add Input -> Datasets -> Your Datasets."
SRC = str(Path(hits[0]).parent.parent)          # folder that contains the tracelens package
WORK, LOG = "/tmp/w", "/kaggle/working/measurements_cpu.jsonl"
WHEELS = f"{COMP}/wheels"
os.makedirs(WORK, exist_ok=True)

def log(**kw):
    kw["ts_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    open(LOG, "a").write(json.dumps(kw) + "\n")

def sh(cmd, timeout=900, env=None, quiet=False):
    t0 = time.time(); e = dict(os.environ); e.update(env or {})
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, env=e)
        rc, out = r.returncode, (r.stdout or "") + (r.stderr or "")
    except subprocess.TimeoutExpired as ex:
        o = ex.stdout.decode() if isinstance(ex.stdout, bytes) else (ex.stdout or "")
        rc, out = 124, o + "\nTIMEOUT"
    if not quiet:
        print(f"$ {cmd[:160]}  -> rc={rc} ({time.time()-t0:.1f}s)\n{out[-1200:]}")
    return rc, out, time.time() - t0

print("COMP:", COMP); print("SRC :", SRC); print("python:", sys.version.split()[0])

In [ ]:
# Diagnostics: confirm the suspected cause (read-only)
sh("python -m pip list 2>/dev/null | grep -i -E 'starlette|fastapi|pydantic|anyio|httpx|^rich|requests' ")
sh(f"ls {WHEELS} | grep -i -E 'starlette|^fastapi|pydantic|anyio|pytest|dirty|inline|httpcore|h11' | tr '\\n' ' '")

In [ ]:
tasks = {}
for l in open(f"{COMP}/tasks.jsonl"):
    t = json.loads(l); tasks[t["instance_id"]] = t

def norm(n): return re.sub(r"[-_.]+", "-", n).lower()
def req_name(r):
    m = re.match(r"\s*([A-Za-z0-9][A-Za-z0-9._-]*)", r); return norm(m.group(1)) if m else None

def test_targets(tp):
    files = sorted(set(re.findall(r"^\+\+\+ b/(\S+)", tp, re.M)))
    files = [f for f in files if re.search(r"(^|/)test_[^/]*\.py$|_test\.py$", f)]
    names = sorted(set(re.findall(r"^\+\s*(?:async )?def (test\w+)", tp, re.M)) |
                   set(re.findall(r"^@@.*@@.*def (test\w+)", tp, re.M)))
    return files, names

def fresh(iid, tag):
    d = f"{WORK}/{iid}_{tag}"; shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    sh(f"tar -xzf {COMP}/snapshots/{iid}.tgz -C {d}", quiet=True)
    repo = d if os.path.isdir(f"{d}/.git") else next((f"{d}/{x}" for x in os.listdir(d) if os.path.isdir(f"{d}/{x}/.git")), None)
    assert repo, f"no .git directory found in snapshot {iid}"
    sh(f"cd {repo} && git config user.email a@b.c && git config user.name a", quiet=True)
    return repo

def apply(repo, text, name):
    p = f"{os.path.dirname(repo)}/{name}.patch"
    open(p, "w").write(text if text.endswith("\n") else text + "\n")
    for cmd in (f"git apply {p}", f"git apply --3way {p}", f"patch -p1 < {p}"):
        if sh(f"cd {repo} && {cmd}", quiet=True)[0] == 0: return True
    return False

KEEP = ("test", "tests", "testing", "dev", "all", "standard")
def poetry_spec(spec):
    if isinstance(spec, dict): spec = spec.get("version", "*")
    if not isinstance(spec, str) or spec.strip() in ("", "*"): return ""
    s = spec.strip()
    m = re.match(r"\^(\d+)(?:\.(\d+))?(?:\.(\d+))?$", s)
    if m:
        a, b, c = (int(x) if x else 0 for x in m.groups())
        return f">={a}.{b}.{c},<{a + 1}" if a > 0 else f">={a}.{b}.{c},<0.{b + 1}"
    m = re.match(r"~(\d+)(?:\.(\d+))?(?:\.(\d+))?$", s)
    if m:
        a, b, c = (int(x) if x else 0 for x in m.groups())
        return f">={a}.{b}.{c},<{a}.{b + 1}" if m.group(2) else f">={a},<{a + 1}"
    return s if re.match(r"[<>=!]", s) else ""

def requirement_list(repo):
    import tomllib
    reqs, me = ["pytest"], None
    p = f"{repo}/pyproject.toml"
    if os.path.exists(p):
        d = tomllib.load(open(p, "rb")); proj = d.get("project", {})
        me = norm(proj["name"]) if proj.get("name") else None
        reqs += proj.get("dependencies", [])
        for k, v in proj.get("optional-dependencies", {}).items():
            if k.lower() in KEEP: reqs += v
        for k, v in d.get("dependency-groups", {}).items():
            if k.lower() in KEEP: reqs += [x for x in v if isinstance(x, str)]
        poetry = d.get("tool", {}).get("poetry", {})
        deps = [poetry.get("dependencies", {})] + [g.get("dependencies", {}) for k, g in poetry.get("group", {}).items() if k.lower() in KEEP]
        for dd in deps:
            for k, spec in dd.items():
                if k.lower() != "python": reqs.append(k + poetry_spec(spec))
    for f in ("requirements.txt", "requirements-dev.txt", "requirements-tests.txt", "tests/requirements.txt"):
        if os.path.exists(f"{repo}/{f}"):
            for l in open(f"{repo}/{f}"):
                l = l.split("#")[0].strip()
                if l and not l.startswith("-"): reqs.append(l)
    return sorted({r for r in reqs if req_name(r) and req_name(r) != me and "@" not in r and "://" not in r})


# ---- mini resolver: pick wheels from the competition wheelhouse WITHOUT needing every transitive dependency ----
try:
    from packaging.requirements import Requirement
    from packaging.specifiers import SpecifierSet
    from packaging.utils import parse_wheel_filename
    from packaging.tags import sys_tags
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
    from pip._vendor.packaging.specifiers import SpecifierSet
    from pip._vendor.packaging.utils import parse_wheel_filename
    from pip._vendor.packaging.tags import sys_tags
import email, zipfile

TAGS = {(t.interpreter, t.abi, t.platform) for t in sys_tags()}
IDX = {}
for w in glob.glob(f"{WHEELS}/*.whl"):
    try: n, v, b, tags = parse_wheel_filename(os.path.basename(w))
    except Exception: continue
    if any((t.interpreter, t.abi, t.platform) in TAGS for t in tags):
        IDX.setdefault(norm(n), []).append((v, w))
print("wheelhouse:", sum(len(x) for x in IDX.values()), "compatible wheels,", len(IDX), "distributions")

_REQ_CACHE = {}
def wheel_requires(whl):
    if whl not in _REQ_CACHE:
        with zipfile.ZipFile(whl) as z:
            mp = next(n for n in z.namelist() if n.endswith(".dist-info/METADATA"))
            _REQ_CACHE[whl] = email.message_from_bytes(z.read(mp)).get_all("Requires-Dist") or []
    return _REQ_CACHE[whl]

def mini_resolve(roots):
    """Greedy: highest wheel satisfying the accumulated specifiers. Dependencies that have no wheel are skipped
    (the notebook's global environment supplies them)."""
    spec, chosen, queue, skipped, notes = {}, {}, list(roots), set(), []
    for _ in range(2000):
        if not queue: break
        try: r = Requirement(queue.pop(0))
        except Exception: continue
        if r.marker and not r.marker.evaluate({"extra": ""}): continue
        n = norm(r.name); spec[n] = spec.get(n, SpecifierSet()) & r.specifier
        cands = [(v, p) for v, p in IDX.get(n, []) if spec[n].contains(v, prereleases=True)]
        if not cands:
            (notes.append(f"no wheel satisfies {n}{spec[n]}") if n in IDX else skipped.add(n)); continue
        best = max(cands)
        if chosen.get(n) == best: continue
        chosen[n] = best; queue += wheel_requires(best[1])
    return chosen, sorted(skipped), notes

def build_env(repo, deps):
    roots = [r for r in requirement_list(repo) if req_name(r) != "pytest"]     # keep the notebook's own pytest
    chosen, skipped, notes = mini_resolve(roots)
    shutil.rmtree(deps, ignore_errors=True); os.makedirs(deps)
    paths = " ".join(shlex.quote(p) for _, p in chosen.values())
    rc, out, _ = sh(f"python -m pip install --no-deps --no-index --target {deps} {paths}", timeout=900, quiet=True) if chosen else (0, "", 0)
    for f in ("imp.py", "telnetlib.py"):
        if os.path.exists(f"{COMP}/docker/{f}"): shutil.copy(f"{COMP}/docker/{f}", deps)
    return {"installed": {k: str(v[0]) for k, v in sorted(chosen.items())}, "skipped_no_wheel": skipped, "notes": notes,
            "pip_rc": rc, "pip_tail": out[-500:] if rc else ""}

def run_tests(repo, files, names, deps, traced, out_json):
    args = " ".join(files) + " -p no:anyio -p no:cacheprovider -q" + (f" -k {shlex.quote(' or '.join(names))}" if names else "")
    env = {"PYTHONPATH": ((SRC + ":") if traced else "") + ":".join([deps, repo, f"{repo}/src"])}
    cmd = (f"cd {repo} && python -m tracelens.run --root {repo} --out {out_json} -- pytest {args}" if traced
           else f"cd {repo} && python -m pytest {args}")
    return sh(cmd, timeout=900, env=env, quiet=True)

In [ ]:
def run_task(iid):
    t = tasks[iid]; files, names = test_targets(t["test_patch"])
    res = {"step": "T0.8c", "instance": iid, "recipe": "mini-resolver wheels first on PYTHONPATH", "test_files": files, "n_named_tests": len(names)}
    repo_pre = fresh(iid, "pre"); res["apply_test_patch"] = apply(repo_pre, t["test_patch"], "test")
    deps = f"{WORK}/{iid}_deps"; t0 = time.time()
    res["env"] = build_env(repo_pre, deps); res["env_s"] = round(time.time() - t0, 1)
    open("/kaggle/working/envs.jsonl", "a").write(json.dumps({"instance": iid, **res["env"]}) + "\n")
    probe = sh(f"cd {repo_pre} && python -c \"import importlib.util as u,sys; "
               f"print({{m: (u.find_spec(m).origin if u.find_spec(m) else None) for m in ('starlette','pydantic','anyio','fastapi','httpx','rich','requests')}})\"",
               env={"PYTHONPATH": ":".join([deps, repo_pre, f"{repo_pre}/src"])}, quiet=True)[1]
    res["import_origins"] = probe.strip()[-500:]
    for tag in ("pre", "post"):
        if tag == "post":
            repo = fresh(iid, "post"); res["apply_gold"] = apply(repo, t["patch"], "gold"); apply(repo, t["test_patch"], "test")
        else:
            repo = repo_pre
        uj = f"{WORK}/{iid}_{tag}.json"
        rc_u, out_u, s_u = run_tests(repo, files, names, deps, False, None)
        rc_t, out_t, s_t = run_tests(repo, files, names, deps, True, uj)
        d = json.load(open(uj)) if os.path.exists(uj) else {}
        oc = {k.split("::")[-1][:60]: v["outcome"] for k, v in d.get("tests", {}).items()}
        res[tag] = {"untraced_rc": rc_u, "untraced_s": round(s_u, 2), "traced_s": round(s_t, 2), "overhead": round(s_t / max(s_u, 1e-6), 2),
                    "n_passed": sum(v == "passed" for v in oc.values()), "n_failed": sum(v == "failed" for v in oc.values()),
                    "n_nodes": len(d.get("nodes", [])), "n_edges": len(d.get("edges", [])), "backend": d.get("backend")}
        if not oc: res[tag]["tail"] = "untraced: " + out_u[-450:] + " || traced: " + out_t[-250:]
    log(**res); return res

def verdict(r):
    return "PASS" if (r["pre"]["n_failed"] > 0 and r["post"]["n_failed"] == 0 and r["post"]["n_passed"] > 0) else "CHECK"

IDS = ["fastapi_14492", "fastapi_9425", "rich_3278", "httpx_3672", "requests_6644"]
results = {}
for iid in IDS:
    try: results[iid] = run_task(iid)
    except Exception as e:
        results[iid] = {"error": repr(e)}; log(step="T0.8c", instance=iid, error=repr(e))

In [ ]:
for iid, r in results.items():
    if "error" in r: print(f"\n=== {iid}: ERROR {r['error']}"); continue
    e = r["env"]
    print(f"\n=== {iid}: {verdict(r)} | env {r['env_s']}s | pip_rc={e['pip_rc']} | gold applied: {r.get('apply_gold')}")
    print("   installed:", {k: v for k, v in e["installed"].items() if k in ("starlette", "pydantic", "pydantic-core", "anyio", "typing-extensions", "httpx", "httpcore", "rich", "requests", "urllib3")})
    print("   no wheel (taken from notebook env):", e["skipped_no_wheel"][:15], "| notes:", e["notes"][:4])
    print("   import origins:", r["import_origins"])
    if e["pip_tail"]: print("   PIP TAIL:", e["pip_tail"].replace("\n", " ")[-400:])
    for tag in ("pre", "post"):
        x = r[tag]; print(f"   {tag}: passed={x['n_passed']} failed={x['n_failed']} nodes={x['n_nodes']} edges={x['n_edges']} overhead={x['overhead']}x")
        if "tail" in x: print("   TAIL:", x["tail"].replace("\n", "\n         "))
json.dump({k: (verdict(v) if "pre" in v else "ERROR") for k, v in results.items()}, open("/kaggle/working/env_recipe_c.json", "w"), indent=1)
print("\nDownload measurements_cpu.jsonl, envs.jsonl and env_recipe_c.json from the Output panel.")